# Lab 01 Challenge: worked solution

Instructor copy (`gpt-4o-mini`, 13 Oct 2026). Tasks 1 and 2 are the core; task 3 is the extension. It repeats the main notebook's first two cells and the pieces of Parts C and F the challenge needs,
so it runs on its own.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Locally (after uv sync) nothing is installed.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/agentic_ai_conversational_agents").exists():
        !git clone -q --depth 1 https://github.com/tatwan/agentic_ai_conversational_agents.git /content/agentic_ai_conversational_agents
    %cd /content/agentic_ai_conversational_agents
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: agentic_ai_conversational_agents


In [2]:
# Config: model names come from models.env; the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name in ("CHAT_MODEL", "REASONING_EFFORT", "JUDGE_MODEL", "EMBED_MODEL"):
    print(f"{name:18}{settings[name] or '(not set)'}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

CHAT_MODEL        gpt-4o-mini
REASONING_EFFORT  (not set)
JUDGE_MODEL       gpt-4o-mini
EMBED_MODEL       text-embedding-3-small
OPENAI_API_KEY    found in .env


## Setup: the pieces from the lab

The question, the instructions, Part C's `get_policy`, Part F's model and agent, unchanged, and the
challenge's `run_and_count`.

In [3]:
import json

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

question = "How long does a digital download link last?"
SYSTEM = "You help Chinook support reps. Check the policy before you answer."

CORPUS = ROOT / "data" / "corpus"
POLICIES = sorted(path.stem.removeprefix("policy_") for path in CORPUS.glob("policy_*.md"))


def get_policy(name: str) -> str:
    """Return the full text of one Chinook support policy.

    Call it before answering any question about shop rules.

    Args:
        name: Policy name, e.g. "refunds" or "escalations".
    """
    path = CORPUS / f"policy_{name}.md"
    if not path.exists():
        return f"No policy named {name!r}. Known: {', '.join(POLICIES)}."
    return path.read_text(encoding="utf-8")


policy_tool = tool(get_policy, parse_docstring=True)
model = ChatOpenAI(
    model=settings["CHAT_MODEL"],
    use_responses_api=True,
    reasoning_effort=settings["REASONING_EFFORT"] or None,
)
agent = create_agent(model, tools=[policy_tool], system_prompt=SYSTEM)


def run_and_count(each_agent, text, runs=1):
    """Run one question; print the model calls, the first call's input tokens and the policy names sent."""
    for run in range(1, runs + 1):
        result = each_agent.invoke({"messages": [{"role": "user", "content": text}]})
        # the model's own messages carry usage_metadata; a message a middleware writes has none
        model_calls = [m for m in result["messages"] if m.type == "ai" and m.usage_metadata]
        names = [call["args"].get("name") for m in model_calls for call in m.tool_calls]
        first_input = model_calls[0].usage_metadata["input_tokens"] if model_calls else 0
        print(f"run {run}: {len(model_calls)} model calls, first input {first_input} tokens, names {names}")
        print("   ", result["messages"][-1].text[:150].replace("\n", " "))

## 1. Put the names in the description

A docstring has to be written out, so the 20 names are typed into the `Args:` line.

In [4]:
def get_policy_listed(name: str) -> str:
    """Return the full text of one Chinook support policy.

    Call it before answering any question about shop rules.

    Args:
        name: Policy name, exactly one of: accounts, cancellations, damaged_media, data_retention, digital_downloads, escalations, exchanges, gift_cards, international, library, payment, preorders, privacy, refunds, returns, shipping, student_discount, support_hours, support_reps, warranty.
    """
    return get_policy(name)


listed_agent = create_agent(model, tools=[tool(get_policy_listed, parse_docstring=True)], system_prompt=SYSTEM)

for label, each_agent in (("get_policy", agent), ("get_policy_listed", listed_agent)):
    print(label)
    run_and_count(each_agent, question, runs=3)

get_policy


run 1: 3 model calls, first input 96 tokens, names ['digital downloads', 'digital_downloads']
    A digital album download link lasts for 90 days after purchase. If needed, the customer can request a new link, which will last for 7 days. The downlo


run 2: 2 model calls, first input 96 tokens, names ['digital_downloads']
    A digital album link is available for download for 90 days after purchase. After that, the link expires. Customers can request one new link that lasts


run 3: 3 model calls, first input 96 tokens, names ['digital-downloads', 'digital_downloads']
    A digital album download link is available for 90 days after purchase. If needed, customers can request a new link, which will last for 7 days. The do
get_policy_listed


run 1: 2 model calls, first input 141 tokens, names ['digital_downloads']
    A digital download link lasts for 90 days after purchase. After that, it expires, but you can request one new link, which lasts for an additional 7 da


run 2: 2 model calls, first input 141 tokens, names ['digital_downloads']
    A digital album is available for download for 90 days after purchase. After that, the link expires. Customers can request one new link, which lasts fo


run 3: 2 model calls, first input 141 tokens, names ['digital_downloads']
    A digital album link is available for download for 90 days after purchase. After that, the link expires, but you can request one new link, which lasts


With the names listed, the model asked for `digital_downloads` at once in all three runs: two model calls each.
With `get_policy`, two runs of three guessed a name first (`digital downloads`, `digital-downloads`) and needed three.
The first call's input was 141 tokens against 96: the 20 names cost 45 tokens on every call of every run. With 20
policies that is cheap. With hundreds of documents the list would cost more than it saves and would not fit in a
description; then the copilot searches for the right document instead, as a later lab does.

## 2. Check the source, not only the answer

In [5]:
vinyl = ("A customer's vinyl arrived warped, and it is the second time for the same title. "
         "What should the rep offer?")
result = agent.invoke({"messages": [{"role": "user", "content": vinyl}]})
for message in result["messages"]:
    tool_calls = [call["args"] for call in getattr(message, "tool_calls", [])]
    print(f"{message.type:5} |", tool_calls or message.text[:100].replace("\n", " "))
print()
print(result["messages"][-1].text)

human | A customer's vinyl arrived warped, and it is the second time for the same title. What should the rep
ai    | [{'name': 'refunds'}]
tool  | # Refunds  Refunds go back to the original payment method. They take 5 to 10 business days after sup
ai    | For a customer whose vinyl arrived warped for the second time, the rep should offer a refund to the 

For a customer whose vinyl arrived warped for the second time, the rep should offer a refund to the original payment method, which typically takes 5 to 10 business days after approval.


The rule is in `policy_damaged_media.md`: *If the same title arrives damaged twice, offer a refund instead of a third
copy.* The model read `policy_refunds.md`, which says how refunds are paid (original payment method, 5 to 10 business
days) and nothing about damaged copies. The answer, a refund, matches the real rule, but nothing the model read
supports it: it is right by luck.
The name `refunds` matches the answer the model expected; `damaged_media` shares no word with *warped vinyl*.

Now five runs with each tool: Part F's `get_policy` and task 1's `get_policy_listed`.

In [6]:
for label, each_agent in (("get_policy", agent), ("get_policy_listed", listed_agent)):
    print(label)
    run_and_count(each_agent, vinyl, runs=5)

get_policy


run 1: 2 model calls, first input 110 tokens, names ['refunds']
    In this case, since the customer received the same warped vinyl for the second time, the rep should offer a full refund to the original payment method


run 2: 2 model calls, first input 110 tokens, names ['refunds']
    For a customer whose vinyl arrived warped for the second time, the representative should offer a refund. Refunding the original payment method would b


run 3: 2 model calls, first input 110 tokens, names ['refunds']
    In this situation, since the customer has received a warped vinyl for the second time, the representative should offer a full refund to the original p


run 4: 2 model calls, first input 110 tokens, names ['refunds']
    In this situation, the representative should offer the customer a full refund back to their original payment method since the item was defective (warp


...

run 1: 2 model calls, first input 155 tokens, names ['damaged_media']
    For a customer whose vinyl arrived warped for the second time, the rep should offer a refund instead of sending a third copy. They should also remind 


run 2: 2 model calls, first input 155 tokens, names ['damaged_media']
    The rep should offer a refund since the customer’s vinyl has arrived warped for the second time. They should also photograph the damage and inform the


run 3: 2 model calls, first input 155 tokens, names ['damaged_media']
    The rep should offer the customer a refund instead of a third copy, since this is the second time the same title has arrived warped. They should also 


run 4: 2 model calls, first input 155 tokens, names ['damaged_media']
    If a customer's vinyl has arrived warped for the second time, the rep should offer a refund instead of a third copy. They should also ask the customer


run 5: 2 model calls, first input 155 tokens, names ['damaged_media']
    For a vinyl that arrived warped for the second time, the rep should offer the customer a refund instead of sending a third copy. Be sure to follow the


With `get_policy` the model read `refunds` in 5 of 5 runs (6 of 6 with the run above) and offered a refund each
time: the right answer, supported by nothing it read. With `get_policy_listed` it read `damaged_media` in 5 of 5 and
gave the supported answer: a refund instead of a third copy, with the photograph of the damage the policy asks for. (In
an earlier run of this notebook it read `damaged_media` in 4 of 5; the fifth read `exchanges` and offered an exchange,
which is wrong for a second damaged copy.) Listing the names lets the model pick the right file more often, but it
still picks by matching words. Searching the policies by meaning, in a later lab, is the next step.

## 3. Extension: cap the loop

In [7]:
from langchain.agents.middleware import ModelCallLimitMiddleware

capped = create_agent(model, tools=[policy_tool], system_prompt=SYSTEM,
                      middleware=[ModelCallLimitMiddleware(run_limit=1)])
result = capped.invoke({"messages": [{"role": "user", "content": question}]})
for message in result["messages"]:
    tool_calls = [call["args"] for call in getattr(message, "tool_calls", [])]
    print(f"{message.type:5} |", tool_calls or message.text[:90].replace("\n", " "))
print("last message usage:", result["messages"][-1].usage_metadata)

human | How long does a digital download link last?
ai    | [{'name': 'digital_downloads'}]
tool  | # Digital downloads  A digital album is available for download for 90 days after purchase.
ai    | Model call limits exceeded: run limit (1/1)
last message usage: None


A limit of 3 lets a run with one wrong name finish: one call for the wrong name, one for the right name, one for the
reply.

In [8]:
capped3 = create_agent(model, tools=[policy_tool], system_prompt=SYSTEM,
                       middleware=[ModelCallLimitMiddleware(run_limit=3)])
result = capped3.invoke({"messages": [{"role": "user", "content": question}]})
model_calls = [m for m in result["messages"] if m.type == "ai" and m.usage_metadata]   # the model's own
print(len(model_calls), "model calls")
print(result["messages"][-1].text)

3 model calls
A digital album link is available for download for **90 days** after purchase. After that, the link expires. If needed, customers can request a new link, which lasts for an additional **7 days**. 

The downloads are tied to the email used for the purchase, and no refunds are available unless the file is corrupt.


With `run_limit=1` the run stopped after one model call. That call asked for `digital_downloads`; the agent ran the
tool and returned the policy, but the model never saw it. The last message, *Model call limits exceeded: run limit
(1/1)*, has no `usage_metadata` because the middleware wrote it, not the model, so `run_and_count` would not count it
as a model call. A rep would read it as the copilot's reply, so a limit that low breaks ordinary questions.

With `run_limit=3` the run finished in three model calls, the most the limit allows: a model that keeps asking stops
at the third call.